# SETU - Bengali Hallucination Mitigation (Full Real-Model Run)

**Thesis CSE-98** | Kazi Tajrian Mostafa & Sayed Raisul Alam Raihan | Supervisor: Mr. Ratul Barua

**Kaggle settings (right panel):**
- Accelerator: **GPU T4 x2**
- Internet: **ON**

**Repo setup:** Cell 1 nijei chesta kore git clone, fail korle `/kaggle/input` e uploaded zip khuje ney.
Taito duto path e kaj korbe - repo public hok ba private, kono problem nai.

Run order: Cell 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7

In [ ]:
# ============================================================
# CELL 1: Setup - repo clone (na paile zip/input theke ney)
# ============================================================
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu scikit-learn rank-bm25 datasets rouge-score 2>/dev/null

import os, glob, subprocess, sys, shutil

REPO = "A-Selective-Triage-and-Correct-Framework-for-Hallucination-Mitigation-in-Bng-SLM-CSE-98"
URL  = f"https://github.com/einadid/{REPO}.git"
ROOT = "/kaggle/working"

def has_code(p):
    return os.path.isfile(os.path.join(p, "src", "pipeline", "setu_pipeline.py"))

repo_path = None

# --- try 1: git clone (repo public holei cholbe) ---
if os.path.isdir(os.path.join(ROOT, REPO)):
    shutil.rmtree(os.path.join(ROOT, REPO), ignore_errors=True)
r = subprocess.run(["git", "clone", "--depth", "1", URL, os.path.join(ROOT, REPO)],
                   capture_output=True, text=True)
if r.returncode == 0 and has_code(os.path.join(ROOT, REPO)):
    repo_path = os.path.join(ROOT, REPO)
    print("SETUP: git clone OK (public repo)")
else:
    print("git clone fail:", (r.stderr or "").strip()[:200])

# --- try 2: /kaggle/input theke (zip ba auto-extracted dataset) ---
if repo_path is None:
    import zipfile
    hits = glob.glob("/kaggle/input/**/src/pipeline/setu_pipeline.py", recursive=True)
    if not hits:
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            print("unzipping:", z)
            try:
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(ROOT)
            except Exception as e:
                print("  skip:", e)
        hits = glob.glob(os.path.join(ROOT, "**", "src", "pipeline", "setu_pipeline.py"), recursive=True)
    if hits:
        repo_path = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
        print("SETUP: code pawa geche /kaggle/input theke")

if repo_path is None:
    raise SystemExit(
        "\n*** Repo setup fail ***\n"
        "Duitar ekta koro:\n"
        "  (a) GitHub repo -> Settings -> public koro, abar ei cell run koro\n"
        "  (b) repo -> Code -> Download ZIP -> Kaggle right panel -> Add Input -> Upload Dataset\n"
    )

os.chdir(repo_path)
sys.path.insert(0, repo_path)
print("Repo root :", repo_path)
print("src/      :", sorted(os.listdir("src")))

In [ ]:
# ============================================================
# CELL 2: Import SETU modules + environment check
# ============================================================
import torch, os, sys, json, datetime

from src.config import SETUConfig
from src.models.slm_generator import SLMGenerator
from src.models.nli_model import MultilingualNLI
from src.retrieval.retriever import BengaliRetriever
from src.pipeline.claim_decomposer import BengaliClaimDecomposer
from src.pipeline.uncertainty_scorer import UncertaintyScorer
from src.pipeline.triage_router import TriageRouter
from src.pipeline.data_driven_corrector import DataDrivenCorrector
from src.pipeline.reasoning_corrector import ReasoningCorrector
from src.pipeline.abstention import AbstentionModule
from src.pipeline.reassembly import ReassemblyModule
from src.pipeline.setu_pipeline import SETUPipeline

print("All SETU modules imported OK")
print("GPU      :", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i} : {p.name} | {p.total_memory/1e9:.1f} GB")

cfg = SETUConfig()
cfg.uncertainty.n_samples = 3          # demo te speed er jonno 3 (full experiment e 7)
print("\nDefaults -> model:", cfg.model.name, "| nli:", cfg.uncertainty.nli_model)
print("Abstention phrase:", cfg.evaluation.abstention_phrase)

In [ ]:
# ============================================================
# CELL 3: Real model - Qwen2.5-1.5B-Instruct 4-bit
# ============================================================
gen = SLMGenerator(cfg.model.name, use_4bit=True)     # OOM hole: "Qwen/Qwen2.5-0.5B-Instruct"

def ask(q, max_new_tokens=200):
    return gen.generate(q, max_new_tokens=max_new_tokens, temperature=0.7)

TESTS = [
    ("বাংলাদেশের রাজধানী কোথায়?", "control - should be correct"),
    ("ঢাকার জনসংখ্যা কত? ২০২২ সালের আদমশুমারি অনুযায়ী বলো।", "numeric - hallucination prone"),
    ("Dhaka city te koto lok thake? Exact number dao.", "code-mixed - weak spot"),
]

results = []
for q, tag in TESTS:
    a = ask(q)
    results.append((q, a, tag))
    print("=" * 72)
    print("[", tag, "]")
    print("Q:", q)
    print("A:", a[:700])
    print()

In [ ]:
# ============================================================
# CELL 4: Retriever + small demo corpus (RAG route chalu korte)
# ============================================================
ret = BengaliRetriever(cfg)          # BGE-M3 download (~2GB, first time)

# Thesis er real corpus: Bengali Wikipedia dump. Demo te choto corpus:
DEMO_CORPUS = [
    "বাংলাদেশের রাজধানী ঢাকা। ঢাকা বাংলাদেশের সর্ববৃহৎ শহর।",
    "২০২২ সালের জনশুমারি অনুযায়ী ঢাকা জেলার জনসংখ্যা প্রায় ১ কোটি ৪৫ লাখ।",
    "ঢাকা শহর বুড়িগঙ্গা নদীর তীরে অবস্থিত।",
    "বাংলাদেশের স্বাধীনতা দিবস ২৬ মার্চ এবং বিজয় দিবস ১৬ ডিসেম্বর।",
    "বাংলাদেশের আয়তন ১,৪৭,৫৭০ বর্গকিলোমিটার।",
    "বাংলাদেশের জাতীয় সংসদ ভবন ঢাকার শেরেবাংলা নগরে অবস্থিত।",
    "বাংলাদেশের জাতীয় ফুল শাপলা এবং জাতীয় পাখি দোয়েল।",
]
# NOTE: ei corpus ta demo - thesis e BenHalluEval / Bengali Wikipedia diye replace hobe.

import numpy as np, faiss
from rank_bm25 import BM25Okapi

emb = ret.embedder.encode(DEMO_CORPUS, normalize_embeddings=True).astype("float32")
idx = faiss.IndexFlatIP(emb.shape[1]); idx.add(emb)
ret.index_bn     = idx
ret.passages_bn  = DEMO_CORPUS
ret.bm25_bn      = BM25Okapi([p.split() for p in DEMO_CORPUS])

for q in ["ঢাকার জনসংখ্যা কত?", "বাংলাদেশের রাজধানী কোথায়?", "Dhaka population koto?"]:
    res = ret.retrieve_with_fallback(q, top_k=3)
    print("Q:", q, "| fallback:", res.get("fallback_used"), "| max_score: %.3f" % res.get("max_score", 0))
    for r in res.get("results", [])[:2]:
        print("   -> [%.3f] %s" % (r["score"], r["passage"][:90]))

In [ ]:
# ============================================================
# CELL 5: Decomposition + Uncertainty + Triage (real model)
# ============================================================
nli = MultilingualNLI(cfg.uncertainty.nli_model)
dec = BengaliClaimDecomposer(slm_generator=gen)
unc = UncertaintyScorer(slm_generator=gen, nli_model=nli, config=cfg)
triage = TriageRouter(config=cfg, slm_generator=gen, retriever=ret)

QUERY = "ঢাকার জনসংখ্যা কত? ২০২২ সালের আদমশুমারি অনুযায়ী বলো।"
draft = gen.generate(QUERY, max_new_tokens=200)
print("DRAFT:", draft)
print("-" * 72)

claims = dec.decompose(draft, method="hybrid")
print(f"ATOMIC CLAIMS ({len(claims)}):")

for i, c in enumerate(claims, 1):
    print(f"\n[{i}] {c}")
    try:
        tr = triage.triage(QUERY, c, method="hybrid")
        print(f"    TRIAGE  -> {tr['label']} (conf={tr['confidence']}, data={tr['data_score']}, reasoning={tr['reasoning_score']})")
    except Exception as e:
        print("    TRIAGE failed:", type(e).__name__, e)
    try:
        us = unc.score_claim(QUERY, c)
        print(f"    UNCERT -> combined={us['combined']:.3f} flagged={us['is_flagged']} "
              f"(sem_ent={us['semantic_entropy']:.3f}, self_cons={us['consistency_raw']:.3f}, verb={us['confidence_raw']:.3f})")
    except Exception as e:
        print("    UNCERT failed:", type(e).__name__, e)

In [ ]:
# ============================================================
# CELL 6: FULL SETU pipelane end-to-end (real model)
# ============================================================
ddc   = DataDrivenCorrector(retriever=ret, slm_generator=gen, config=cfg)
rc    = ReasoningCorrector(slm_generator=gen, nli_model=nli, config=cfg)
abst  = AbstentionModule(config=cfg)
reasm = ReassemblyModule(slm_generator=gen, config=cfg)

pipe = SETUPipeline(
    slm_generator=gen,
    claim_decomposer=dec,
    uncertainty_scorer=unc,
    triage_router=triage,
    data_corrector=ddc,
    reasoning_corrector=rc,
    abstention_module=abst,
    reassembly_module=reasm,
    config=cfg,
)

out = pipe.run("ঢাকার জনসংখ্যা কত? ২০২২ সালের আদমশুমারি অনুযায়ী বলো।", task_type="qa", verbose=False)

print("QUERY        :", out["query"])
print("DRAFT        :", out["draft_answer"][:400])
print()
print("CLAIMS       :", len(out["claims"]))
print("STATS        :", json.dumps(out["stats"], ensure_ascii=False))
print()
print("FINAL ANSWER :", out["final_answer"])
print("\nTime: %.1f s" % out["time"])

In [ ]:
# ============================================================
# CELL 7: Supervisor report auto-generate (file → download → send)
# ============================================================
L = []
L.append("# SETU - Progress Report (Real Model Run on Kaggle)")
L.append("")
L.append("Date       : %s" % datetime.date.today().isoformat())
L.append("Model      : %s (4-bit NF4)" % cfg.model.name)
L.append("Hardware   : %s" % (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))
L.append("Repo       : github.com/einadid/%s" % REPO)
L.append("")
L.append("## Implemented pipeline modules (src/pipeline/)")
for f in sorted(os.listdir("src/pipeline")):
    if f.endswith(".py") and f != "__init__.py":
        L.append("- src/pipeline/%s" % f)
L.append("- src/retrieval/retriever.py  (FAISS BGE-M3 + BM25 + cross-lingual fallback)")
L.append("- src/evaluation/benhallu_score.py  (dual-track BenHalluScore)")
L.append("")
L.append("## Real model output (test questions)")
for q, a, tag in results:
    L.append("")
    L.append("**Q [%s]:** %s" % (tag, q))
    L.append("**A:** %s" % a[:500].replace("\n", " "))
L.append("")
L.append("## SETU end-to-end run")
L.append("")
L.append("- Query: %s" % out["query"])
L.append("- Draft: %s" % out["draft_answer"][:400].replace("\n", " "))
L.append("- Claims: %d | Stats: %s" % (len(out["claims"]), json.dumps(out["stats"], ensure_ascii=False)))
L.append("- Final: %s" % out["final_answer"].replace("\n", " ")[:600])
L.append("")
L.append("## Next (Week 2)")
L.append("- BenHalluEval 12K load kore unified schema e ana")
L.append("- Real Bengali Wikipedia index build (demo corpus replace)")
L.append("- RQ1: semantic entropy vs self-consistency + ECE / AUROC calibration")

report = "\n".join(L)
open("/kaggle/working/SUPERVISOR_REPORT_week1.md", "w").write(report)
print(report)
print("\n\n>>> File saved: /kaggle/working/SUPERVISOR_REPORT_week1.md  (Kaggle Output theke download koro)")